In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# Step 1: Create a sample student dataset
np.random.seed(42)
n = 200

data = pd.DataFrame({
    "Attendance": np.random.randint(40, 101, n),
    "InternalMarks": np.random.randint(10, 101, n),
    "AssignmentScore": np.random.randint(20, 101, n),
    "StudyHours": np.random.randint(1, 8, n)
})

# Generate pass/fail labels for demonstration
data["Result"] = (
    (data["Attendance"] >= 65) &
    (data["InternalMarks"] >= 40) &
    (data["AssignmentScore"] >= 40)
).astype(int)

print("First five records:")
print(data.head())

print("\nPass/Fail distribution:")
print(data["Result"].value_counts())

# Step 2: Separate input features and target
X = data[[
    "Attendance",
    "InternalMarks",
    "AssignmentScore",
    "StudyHours"
]]
y = data["Result"]

# Step 3: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 4: Normalize using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 5: Create the ANN
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the model
model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# Step 6: Predict test-set labels
probabilities = model.predict(X_test, verbose=0).ravel()
y_pred = (probabilities >= 0.5).astype(int)

# Evaluate performance
print("\nAccuracy:",
      round(accuracy_score(y_test, y_pred), 4))

print("Precision:",
      round(precision_score(y_test, y_pred, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, y_pred, zero_division=0), 4))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Fail", "Pass"],
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"]
)

plt.xlabel("Predicted Result")
plt.ylabel("Actual Result")
plt.title("Student Pass/Fail Confusion Matrix")
plt.show()
